# SZiFi pilot: FLAMINGO mock → iMMF + sciMMF catalogues (q>5)

Prepare a few high-|b| tiles from existing **total** maps (coadd + Gaussian beam + NPIPE split A), run GPU SZiFi, write catalogues under `/rds/.../szifi/pilot/`.

No ILC algorithm — only multi-frequency total temperature maps as inputs.


In [ ]:
import os
os.environ.setdefault("SZIFI_ARRAY_BACKEND", "jax")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

from flamingo_mock.szifi.paths import SZiFiPaths
from flamingo_mock.szifi.tiles import prepare_tiles, select_pilot_tile_ids
from flamingo_mock.szifi.run import run_imf_and_scimmf, default_params

paths = SZiFiPaths()
paths.make_dirs("A")
field_ids = select_pilot_tile_ids(n=4, b_min_deg=40.0)
ps, _, _ = default_params(paths, field_ids, split="A")
print("pilot tiles:", field_ids)
print("out_root:", paths.out_root)
print("decouple_type:", ps["decouple_type"], "(pilot skips NaMaster coupling matrix)")


In [ ]:
# Cut multi-frequency tiles + PR4 GAL/PS masks (skip if already on disk)
prepare_tiles(paths, field_ids, split="A", overwrite=False)


In [ ]:
# iMMF + sciMMF on GPU; write q>5 catalogues
written = run_imf_and_scimmf(paths, field_ids, split="A", q_th_final=5.0, tag="pilot")
written


## Visualise detections on truth Compton-$y$

Circles mark catalogue positions; radius = fitted $\theta_{500}$. Full-sky markers + zoomed gnomonic view near the north Galactic pole (pilot tiles 0–3).


In [ ]:
from IPython.display import Image, display
display(Image(filename="../figures/szifi_pilot_ymap_mollview.png"))
display(Image(filename="../figures/szifi_pilot_clusters_immf_zoom.png"))
display(Image(filename="../figures/szifi_pilot_clusters_scimmf_zoom.png"))
display(Image(filename="../figures/szifi_pilot_ymap_gnomonic_circles.png"))
